# Bài 5a (Trang 45) - Online Retail (Germany)
## Câu 12: Tìm tập phổ biến và luật kết hợp bằng FP-Growth (min_sup = 5%, min_conf = 50%), so sánh với Apriori

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
%matplotlib inline

df = pd.read_excel("../Dataset/Online Retail.xlsx")

### Các bước xử lý của câu 1 → 11 (chuẩn bị dữ liệu)

In [3]:
df["Description"] = df["Description"].str.strip()
df = df.dropna(axis=0, subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

df = df[~df["InvoiceNo"].str.contains("C")]

grouped = (df[df["Country"] == "Germany"]
           .groupby(["InvoiceNo", "Description"])["Quantity"]
           .sum())

basket = grouped.unstack().reset_index().fillna(0).set_index("InvoiceNo")

def encode_units(x):
    return 1 if x >= 1 else 0

# Tương đương basket.map(encode_units) nhưng vector hóa cho nhanh; kiểu bool theo khuyến nghị của mlxtend
basket_sets = basket >= 1

basket_sets = basket_sets.drop(columns=["POSTAGE"], errors="ignore")

start = time.time()
frequent_itemsets = apriori(basket_sets, min_support=0.05, use_colnames=True)
apriori_time = time.time() - start

rules = association_rules(frequent_itemsets, num_itemsets=len(basket_sets),
                          metric="confidence", min_threshold=0.5)

rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

support = rules["support"].values
confidence = rules["confidence"].values

### Câu 12: Tìm tập phổ biến và luật kết hợp bằng FP-Growth (min_sup = 5%, min_conf = 50%), so sánh với Apriori

In [4]:
start = time.time()
fp_itemsets = fpgrowth(basket_sets, min_support=0.05, use_colnames=True)
fp_time = time.time() - start
fp_rules = association_rules(fp_itemsets, num_itemsets=len(basket_sets),
                             metric="confidence", min_threshold=0.5)
fp_rules["antecedents"] = fp_rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
fp_rules["consequents"] = fp_rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

In [5]:
print("Số tập phổ biến FP-Growth:", len(fp_itemsets))
print("Số luật FP-Growth:", len(fp_rules))
for i, r in fp_rules.iterrows():
    print(f"Rule {i + 1}: {r['antecedents']} => {r['consequents']} "
          f"(support = {r['support']:.3f}, confidence = {r['confidence']:.3f})")

Số tập phổ biến FP-Growth: 59
Số luật FP-Growth: 8
Rule 1: ROUND SNACK BOXES SET OF4 WOODLAND => ROUND SNACK BOXES SET OF 4 FRUITS (support = 0.131, confidence = 0.536)
Rule 2: ROUND SNACK BOXES SET OF 4 FRUITS => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.131, confidence = 0.833)
Rule 3: WOODLAND CHARLOTTE BAG => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.063, confidence = 0.500)
Rule 4: PLASTERS IN TIN CIRCUS PARADE => PLASTERS IN TIN WOODLAND ANIMALS (support = 0.068, confidence = 0.585)
Rule 5: SPACEBOY LUNCH BOX => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.070, confidence = 0.681)
Rule 6: RED RETROSPOT CHARLOTTE BAG => WOODLAND CHARLOTTE BAG (support = 0.059, confidence = 0.844)
Rule 7: PLASTERS IN TIN WOODLAND ANIMALS => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.074, confidence = 0.540)
Rule 8: PLASTERS IN TIN SPACEBOY => PLASTERS IN TIN WOODLAND ANIMALS (support = 0.061, confidence = 0.571)


In [6]:
# So sánh với Apriori
same_itemsets = set(frequent_itemsets["itemsets"]) == set(fp_itemsets["itemsets"])
key = lambda d: set(zip(d["antecedents"], d["consequents"]))
same_rules = key(rules) == key(fp_rules)
comparison = pd.DataFrame({
    "Apriori": [len(frequent_itemsets), len(rules), round(apriori_time, 3)],
    "FP-Growth": [len(fp_itemsets), len(fp_rules), round(fp_time, 3)],
}, index=["Số tập phổ biến", "Số luật", "Thời gian (s)"])
print("Tập phổ biến giống nhau:", same_itemsets)
print("Tập luật giống nhau:", same_rules)
comparison

Tập phổ biến giống nhau: True
Tập luật giống nhau: True


,Apriori,FP-Growth
Số tập phổ biến,59.000,59.000
Số luật,8.000,8.000
Thời gian (s),0.018,0.022


### Nhận xét
- FP-Growth cho ra **cùng 59 tập phổ biến và 8 luật** như Apriori.
- Dữ liệu nước Đức nhỏ (~450 hóa đơn) nên thời gian chạy của 2 thuật toán gần như tương đương.